In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', 100)

# Load main data
train = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\application_train.csv')
test  = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\application_test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
print(train.dtypes.value_counts())

In [ ]:
target_counts = train['TARGET'].value_counts()
print(target_counts)
print(f"\nDefault rate: {train['TARGET'].mean():.2%}%")

In [ ]:
print(train.columns.tolist())

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(6, 4))

axes[0].bar(['Repaid(0)', 'Defaulted(1)'], target_counts.values, color=['green', 'red'])
axes[0].set_title('Distribution of Target Variable')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1000, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['Repaid(0)', 'Defaulted(1)'], autopct='%1.1f%%', colors=['green', 'red'])
axes[1].set_title('Proportion of Target Variable')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
missing=pd.DataFrame({
    'column': train.columns,
    'missing_count': train.isnull().sum().values,
    'missing_percentage': (train.isnull().sum().values / len(train)) * 100
}).sort_values(by='missing_percentage', ascending=False)

missing = missing[missing['missing_percentage'] > 0]

print(f"Columns with missing values: {len(missing)}")
print(f"Columns with >50% missing values: {(missing['missing_percentage'] > 50).sum()}")
print(f"Columns with >80% missing values: {(missing['missing_percentage'] > 80).sum()}")

print("Top 20 worst columns")
print(missing.head(20).to_string(index=False))

In [ ]:
import missingno as msno
fig, ax = plt.subplots(figsize=(16, 8))
cols_high_missing = missing[missing['missing_percentage'] > 20]['column'].tolist()

msno.bar(train[cols_high_missing], ax=ax, color='skyblue', fontsize=8)
ax.set_title('Missing Values in Columns with >20% Missing', fontweight='bold')
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
msno.heatmap(train[cols_high_missing[:30]],figsize=(14,8))
plt.title("Missing Value Correlation Heatmap")
plt.savefig('missing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cat_cols=train.select_dtypes('object').columns
print(f"Categorical columns: {len(cat_cols)}")
print(cat_cols.tolist())

In [ ]:
for col in cat_cols:
    print(f"{col:<40} {train[col].nunique()} unique values")

In [ ]:
important_cats = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 
                  'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'OCCUPATION_TYPE']

fig, axes = plt.subplots(2, 3, figsize=(20, 25))
axes = axes.flatten()

for i, col in enumerate(important_cats):
    default_counts = train.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    bars=axes[i].barh(default_counts.index, default_counts.values * 100, color=plt.cm.RdYlGn_r(default_counts.values/default_counts.max()))
    axes[i].set_xlabel('Default Rate (%)')
    axes[i].set_title(f'Default Rate by {col}')
    for bar in zip(bars, default_counts.values * 100):
        width = bar[0].get_width()
        axes[i].text(width + 0.5, bar[0].get_y() + bar[0].get_height()/2, f'{width:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('default_rate_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
num_cols = train.select_dtypes(['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'TARGET' and c != 'SK_ID_CURR']
print(f"Numerical columns: {len(num_cols)}")

In [ ]:
key_num = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 
           'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(key_num):

    default_0 = train[train['TARGET'] == 0][col].dropna()
    default_1 = train[train['TARGET'] == 1][col].dropna()
    
    axes[i].hist(default_0, bins=50, alpha=0.6, label='Repaid', color='#2ecc71', density=True)
    axes[i].hist(default_1, bins=50, alpha=0.6, label='Defaulted', color='#e74c3c', density=True)
    axes[i].set_title(f'{col}', fontsize=10)
    axes[i].legend()
    axes[i].set_ylabel('Density')

plt.suptitle('Distribution of Key Features by Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/05_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train['AGE_YEARS'] = -train['DAYS_BIRTH'] / 365
print(f"Age range: {train['AGE_YEARS'].min():.1f} to {train['AGE_YEARS'].max():.1f} years")

In [ ]:
print(f"\nDAYS_EMPLOYED max value: {train['DAYS_EMPLOYED'].max()}")
print(f"Count of 365243: {(train['DAYS_EMPLOYED'] == 365243).sum()}")

In [ ]:
train['DAYS_EMPLOYED_ANOM'] = (train['DAYS_EMPLOYED'] == 365243)
train['DAYS_EMPLOYED'] = train['DAYS_EMPLOYED'].replace(365243, np.nan)
print(f"\nAnomaly flag created. {train['DAYS_EMPLOYED_ANOM'].sum()} rows flagged.")

In [ ]:
print(train.groupby('DAYS_EMPLOYED_ANOM')['TARGET'].mean()*100)

In [ ]:
corr_with_target = train.select_dtypes(['int64', 'float64']).corr()['TARGET'].drop('TARGET')
corr_with_target = corr_with_target.dropna().sort_values()

top_pos = corr_with_target.tail(20)
top_neg = corr_with_target.head(20)
top_corr = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(10, 14))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in top_corr.values]
ax.barh(top_corr.index, top_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with TARGET')
ax.set_title('Top 40 Features by Correlation with Default', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/06_correlation_target.png', dpi=150, bbox_inches='tight')
plt.show()

print("Most positive (→ higher default):", top_pos.tail(5).index.tolist())
print("Most negative (→ lower default):", top_neg.head(5).index.tolist())

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(ext_cols):
    axes[i].hist(train[train['TARGET']==0][col].dropna(), bins=40, 
                  alpha=0.6, label='Repaid', density=True, color='#2ecc71')
    axes[i].hist(train[train['TARGET']==1][col].dropna(), bins=40, 
                  alpha=0.6, label='Defaulted', density=True, color='#e74c3c')
    axes[i].set_title(f'{col}\n(Corr: {train[col].corr(train["TARGET"]):.3f})')
    axes[i].legend()

plt.suptitle('External Credit Scores — Most Predictive Features!', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/07_ext_sources.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
supplementary = {
    'bureau':           r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\bureau.csv',
    'bureau_balance':   r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\bureau_balance.csv',
    'prev_application': r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\previous_application.csv',
    'pos_cash':         r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\POS_CASH_balance.csv',
    'credit_card':      r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\credit_card_balance.csv',
    'installments':     r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\installments_payments.csv',
}

for name, path in supplementary.items():
    df = pd.read_csv(path, nrows=5) 
    full_df = pd.read_csv(path)
    print(f"\n{'='*50}")
    print(f" {name}: {full_df.shape[0]:,} rows × {full_df.shape[1]} cols")
    print(f"   Columns: {full_df.columns.tolist()}")
    missing_pct = full_df.isnull().mean().mean() * 100
    print(f"   Overall missing: {missing_pct:.1f}%")
    del full_df

In [ ]:
eda_summary = {
    'train_rows': len(train),
    'train_cols': train.shape[1],
    'test_rows':  len(test),
    'default_rate': f"{train['TARGET'].mean()*100:.2f}%",
    'cols_with_missing': (train.isnull().sum() > 0).sum(),
    'cols_missing_50pct': (train.isnull().mean() > 0.5).sum(),
    'categorical_cols': len(train.select_dtypes('object').columns),
    'numerical_cols': len(train.select_dtypes(['int64','float64']).columns),
}

print("EDA SUMMARY")
print("="*40)
for k, v in eda_summary.items():
    print(f"  {k:<30}: {v}")